# Conditional analysis from SCWF coefficient stores

This notebook consumes the **coefficient-only** output of the revised pipeline.

Use the saved long-form `CoefficientStore` rows, not interval-averaged `Spectra`, `Sfuncs`, or `WaveletMoments`.

Second-order conditional spectra are estimated from

`P_hat = < |a|^2 / response_energy_integral >`

inside each scale bin and conditioning subset.

Scale-normalized higher-order moments are estimated from

`M_hat_q = < |a|^q / tau_equiv_seconds^(q/2) >`.

The long-form store already contains everything needed:
- local projected scales: `l_mag`, `l_ell`, `l_xi`, `l_lambda`
- local angles: `thetas`, `phis`
- normalization columns: `tau_equiv_seconds`, `frequency_hz`, `response_energy_integral`
- row-level coefficients and projected amplitudes

Directional buckets are reconstructed from the saved angles and the stored bucket conditions.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from cond_analysis_scwf import load_and_merge_coefficient_stores, reduce_conditional_rows

In [ ]:
# Edit this glob to point to your saved first-pass interval pickles
PATHS = r'C:\Users\nokni\work\WIND_3D\data\3_sec\WIND\*\final_vs8\*.pkl'

# Load one merged long-form dataframe across all intervals/scales
df = load_and_merge_coefficient_stores(PATHS, bucket='ell_all')
df.shape

In [ ]:
df.head()

## Example 1: conditional trace spectra for wave-vector anisotropy

Use trace-like scalar coefficient magnitudes across different directional buckets.
Examples: `W_B_vel_mag`, `W_B_nT_mag`, `W_V_mag`, `W_Zp_mag`, `W_Zm_mag`.

In [ ]:
scale_edges = np.geomspace(1.0, 1.0e4, 41)

psd_ell_perp = reduce_conditional_rows(
    df,
    bucket='ell_perp',
    value_keys=['W_Zp_mag', 'W_Zm_mag', 'W_B_vel_mag', 'W_V_mag'],
    cond_var='compress_simple',
    qorder=[2.0],
    normalization='psd',
    scale_bin_edges_di=scale_edges,
    constraints={'coi_mask': (0.5, None), 'is_effective_level': (0.5, None)},
    nquant=10,
    min_count=25,
)
psd_ell_perp.head()

## Example 2: scale-normalized higher-order moments

Use the same trace columns or projected amplitudes such as `V_par`, `V_perp`, `Zp_par`, `Zp_perp`, `B_vel_par`, `B_vel_perp`.

These are the quantities that can be interpreted as **structure-function surrogates** with the correct physical units `X^q`.

In [ ]:
mom_ell_par = reduce_conditional_rows(
    df,
    bucket='ell_par',
    value_keys=['V_par', 'V_perp', 'Zp_par', 'Zp_perp'],
    cond_var='sig_c_ts',
    qorder=[1.0, 2.0, 3.0, 4.0],
    normalization='scale_normalized',
    scale_bin_edges_di=scale_edges,
    constraints={'coi_mask': (0.5, None), 'is_effective_level': (0.5, None)},
    nquant=10,
    min_count=25,
)
mom_ell_par.head()

## Notes

- For **wave-vector anisotropy**, compare the same trace observable across `ell_perp`, `Ell_perp`, and `ell_par`.
- For **component scaling**, compare projected quantities within a fixed bucket.
- Do not interpret `B_xi` and `B_lambda` as independent magnetic polarization diagnostics when the local `xi` direction is defined from the magnetic perpendicular fluctuation itself.
- Fit scaling exponents only after inspecting count support and the spread of actual local projected scales inside each bin.